In [151]:
import pandas as pd
import re

In [152]:
DIRECTORY = 'qwen2.5-coder_32b'
FILE_PATH = f'{DIRECTORY}/sv_results_{DIRECTORY}_1.csv'

In [153]:
data = pd.read_csv(FILE_PATH)

In [154]:
data['iverilog_output'].value_counts()

iverilog_output
OK                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [155]:
def extract_code(output: str, iverilog: str)-> str:
    '''
    Auxiliar function to extract the systemverilog code from the LLM output.
    This is just a safeguard in case the model decides to putput something else than SystemVerilog code.
    '''
    is_valid = iverilog

    if iverilog == 'OK':
        has_testing_in_code = re.findall(r'\b(assert|property)\b', output)
        print(output, bool(has_testing_in_code))
        print('---------------------------------------------------------------')
        if not has_testing_in_code:
            is_valid = 'NON_VALID_OUTPUT'

    return is_valid

In [156]:
data['iverilog_output'] = data.apply(
    lambda row: extract_code(row['generated_code'], row['iverilog_output']), 
    axis=1
)

module case_example1(input logic [1:0] a, output logic y);
    always_comb begin
        case(a)
            2'b00: y = 0;
            2'b01: y = 1;
            2'b10: y = 1;
            2'b11: y = 0;
            default: y = 0; // This default case is technically redundant since a is 2 bits wide
        endcase
    end

    // Property to check the output behavior for each input combination
    property p_case_behavior;
        @(posedge a) disable iff ($isunknown(a))
            (a == 2'b00 |-> y == 0) and
            (a == 2'b01 |-> y == 1) and
            (a == 2'b10 |-> y == 1) and
            (a == 2'b11 |-> y == 0);
    endproperty

    // Assertion to check the property
    a_case_behavior: assert property(p_case_behavior) else $error("Output does not match expected behavior for input %b", a);

endmodule True
---------------------------------------------------------------
module case10(input logic [2:0] pattern, output logic match);
    always_comb begin
        case(pattern)
 

In [157]:
data['iverilog_output'].value_counts()

iverilog_output
OK                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [103]:
data['iverilog_output'] = data['iverilog_output'].replace(['NO_SV_MODULE_FOUND', 'CODE_BLOCK_NOT_FOUND'], 'NON_VALID_OUTPUT')

In [104]:
data.to_csv(FILE_PATH, index=False)